# 第22章：writing-for-agents——为Agent编写可靠的文档

## 22.1 资料范围与核心目标

本篇依据Matt Pocock的skills仓库中 `skills/productivity/writing-for-agents/` 目录整理，包含通用写作参考 `SKILL.md`、技能机制分支 `SKILL-MECHANICS.md` 和界面元数据 `agents/openai.yaml`。它属于本课程之后的补充阅读，并非尚硅谷原课章节。

源仓库：[mattpocock/skills](https://github.com/mattpocock/skills)。本次固定版本 `7739eaed7f8f3915437a8def198a6d0d7206dbc7`，读取日期2026-10-29。

- [SKILL.md原文](https://github.com/mattpocock/skills/blob/7739eaed7f8f3915437a8def198a6d0d7206dbc7/skills/productivity/writing-for-agents/SKILL.md)
- [SKILL-MECHANICS.md原文](https://github.com/mattpocock/skills/blob/7739eaed7f8f3915437a8def198a6d0d7206dbc7/skills/productivity/writing-for-agents/SKILL-MECHANICS.md)
- [agents/openai.yaml原文](https://github.com/mattpocock/skills/blob/7739eaed7f8f3915437a8def198a6d0d7206dbc7/skills/productivity/writing-for-agents/agents/openai.yaml)

作者关心的是执行过程是否稳定：换一份输入以后，Agent仍按正确方法读取、判断、操作和验收，输出则应随输入变化。需要写好的对象包括技能文件、AGENTS.md、CLAUDE.md，以及经链接按需读取的参考文档。

本篇用中文解释原文概念；与订单案例结合的片段标为“解释示例”。这些是Notebook中的阅读材料，没有改动当前仓库的Agent规则或安装技能。

## 22.2 Context pointer：决定资料什么时候被读到

上下文指针是一条已在Agent上下文里的引用：它既指明外部资料是什么，也说明什么情况需要读取。技能description、AGENTS.md里“修改数据库时读迁移说明”的一句话，都属于这种对象。

目标文档写得再好，如果入口只说“这里还有一些资料”，Agent也可能不知道何时该读。原文把这种入口不明确造成的不稳定视为一种设计缺陷：先强化指针的触发条件，仍无效时再考虑把必需内容放回主文件。

指针有两项职责：概括资料用途，列出真正不同的触发分支。一个分支是文档实际处理的一种情形；为同一情形堆“总结、概括、归纳”等近义词，不会增加有意义的覆盖。

### 解释示例：从泛泛链接改为条件入口

```markdown
订单核对：处理订单、支付和履约导出时，读取 references/trade-rules.md，
确认清洗顺序、支付标签和统计口径后再计算。
```

这段能让Agent判断任务是否命中，也知道读取后要解决什么问题。入口宜把关键概念提前，每个真实分支留一个清楚触发，删去正文已经说明的身份介绍。入口常驻上下文，冗余成本比按需正文更经常发生。

## 22.3 两种负担：上下文成本与人的记忆成本

| 概念 | 成本由谁承担 | 典型内容 |
| --- | --- | --- |
| Context load，上下文负担 | Agent的上下文窗口与注意力 | 始终加载的description、项目规则、入口行 |
| Cognitive load，认知负担 | 需要记住工具入口的人 | 有哪些技能、何时调用、去哪找资料 |

将参考资料放到指针后，可以让完整正文按需加载，但指针本身仍占成本。完全没有入口的文件则需要人记得提供，主要消耗人的认知负担。

原文没有要求把人的记忆成本一律降到零。某些操作应由人判断何时发起，保留手动入口有价值；那些人不需要反复判断的机械选择，则可以交给明确的自动触发。设计要问成本换来了什么，而不只是比谁的文件更短。

## 22.4 信息层级：步骤、文件内参考、按需参考

Agent文档常混合两类内容：steps是按顺序要做的动作，reference是执行时需要查的定义、约束与事实。纯流程、纯参考或二者混合都可以，重要的是位置安排。

| 层级 | 放什么 | 何时需要 |
| --- | --- | --- |
| 文件内步骤 | 主要执行路径与顺序 | 开始任务时立即掌握 |
| 文件内参考 | 与当前路径紧密相邻的规则 | 执行到相关判断时查看 |
| 按需展开的参考 | 独立文件或外部资料，由指针引导 | 只有部分分支会用到 |

并非所有文档都必须有步骤。审查规则可以是一组平级参考，只要各条确实需要应用；不能因为没有线性流程就把它当坏结构。

### 渐进式展开的判断方法

每条分支都需要的核心约束留在主文件；只有少数分支才用的长规则放在条件指针后。主文件过满会让步骤被参考信息埋住，拆得过细又会把必需内容藏起来。

例如先明确“读输入→清洗→计算→输出→复核”，对只在日期冲突时用的长解释，可以在冲突分支引用。分层的主要目的是让任务顺序可辨、每条路径带上合适信息，节省token是伴随收益。

## 22.5 Co-location：同一概念的定义、规则和例外放一起

信息层级决定内容放主文件还是外部；同位组织决定在同一文件里哪些内容相邻。一个概念的定义、适用条件、边界和例外，应尽量放在同一标题下，读者找到规则时也能看到限制。

解释示例：支付标签的优先级、互斥关系、不能认定重复扣款的证据边界放在同节，比规则在开头、边界在文末、例外在附录更容易完整应用。

原文区分两类问题：duplication是同一含义重复写多处；scattering是一个含义被拆碎到多处。前者增加维护与强调，后者让Agent可能只读到一半。

### Sprawl：即使没有重复也可能过长

每句都有用，不代表所有内容都应挤进同一条执行路径。过长会稀释注意力，也让每一行更难持续保持相关。处理方式是沿信息层级下移参考，或按分支/顺序组织，让一次任务只携带所需内容，而不是盲目删除关键约束。

## 22.6 完成标准：让Agent知道什么时候真的做完

每一步都需要completion criterion，即可判断完成与未完成的条件。原文从两个维度分析：clarity是界限清不清楚，demand是要求覆盖到什么程度。

| 写法 | 问题 | 更可核对的写法 |
| --- | --- | --- |
| 理解数据 | 看不出理解是否充分 | 列出全部输入表、主键、关联键和统计粒度 |
| 检查修改 | 范围不明确 | 每个改动文件对应需求，并记录受影响场景 |
| 报告生成完成 | 文本出现不等于文件可用 | 指定文件已保存，数字与计算JSON一致，未知项有标记 |

表中的改写是解释示例。强标准同时可检查且覆盖完整，而不只是要求产出一个看似完成的清单。

原文用legwork描述标准带出的深入查找：要求每个变更模型都有交代，会促使Agent寻找遗漏；“生成一份清单”可能列几项就结束。纯参考文档也可设“所有适用规则都已逐项检查”的覆盖要求，不限于有步骤的流程。

## 22.7 过早结束与真正的上下文边界

当标准只是“理解到了”，后面可见的交付步骤可能让Agent急于往后走。原文将这种情况称为premature completion；后续步骤对注意力的牵引是post-completion steps带来的影响。

处理优先顺序：先把当前完成条件写清楚，这是局部且便宜的修正。只有标准无法进一步客观化，并实际观察到赶进度时，再考虑拆开执行顺序，使当前执行者暂时看不到后续任务。

原文强调真正隐藏后续步骤需要实际上下文边界，例如交接或委派时只传当前任务；在同一上下文再读另一文件，并不会让已看到的后续步骤消失。本篇只是解释其原理，本次整理笔记没有因这段资料发起委派。

拆分会增加上下文入口或人的记忆成本，因此不能看到长文就拆。反过来，把原本隔离的顺序强行合并，也可能让后续交付干扰当前调查。

## 22.8 Leading word：用共享概念稳定行为

Leading word指一个紧凑、模型已有语义基础的概念词，反复出现时帮助Agent指向同一类行为。原文举了lesson、fog of war、tracer bullets等词，并用tight压缩速度、确定性与开销的一组要求，用red把测试门槛连到可观察的失败状态。

它在两个位置起作用：正文中帮助执行时提取相同行为；指针里帮助任务用词和文档用词建立联系。重点是复用短概念，而不是在多个地方重复整段定义。

已有词汇能调用模型已经学过的语义；自创术语也能用，但要支付解释成本。词必须能实际区分行为，泛泛“认真、全面”如果不改变默认表现，仍可能无效。

### 解释示例：先定义，再统一引用

将“查到的证据、可能的解释、下一步建议”明确区分为事实/假设/建议，此后报告和模板沿用同一组术语。它们需要定义清楚并贯穿一致，不能只是给一串模糊要求换个新名字。

## 22.9 正向表述与必要的硬约束

原文认为，反复点名禁止行为可能让这些行为更显眼，因此更倾向直接描述目标动作。例如写“注释用一行说明必要原因”，比围绕不要写哪些冗长注释更能把注意力放在目标形式。

这是作者提出的写作判断，不是保证所有模型每次如此执行的定律，仍需在真实任务中观察。

必要的硬边界可以保留，同时写清应该怎样做。解释示例：未知负责人填“待确认”，并保留“没有依据不补造负责人”的约束；遇到服务错误写失败范围，并保留“不能把失败当零结果”。目标行为与边界共同指向可执行处理。

## 22.10 Pruning：删减重复、缓存与失效规则

原文从四个方面检查每一行是否值得保留。

**单一事实来源。** 同一业务含义尽量有一个权威位置。修改已支付口径时只改规则资料，其他流程引用它，减少多处定义相互冲突。关键概念词可以重复，完整含义不必每处重述。

**环境也是事实来源。** package.json脚本、配置、目录和--help已经给出的内容，文档重复抄写就形成缓存。只有查找昂贵时才值得缓存；更值得写的是环境里看不出的原因、团队隐含约定和踩坑条件。

**相关性随时间变化。** 一行可能从来不属于任务，也可能规则变化后过时。只增加不移除会形成sediment，像沉积层一样遮住仍有效的要求。整理时逐句确认它现在是否影响当前文档的用途。

**No-op。** 如果一句话对模型默认行为没有任何改变，保留它只增加成本。是否无效与模型默认表现有关，应通过实际执行比较，而不只靠人争论。无效时可删整句，不是仅把词再修短。

## 22.11 技能机制：自动发现与手动调用的取舍

SKILL-MECHANICS把通用文档原则应用到技能入口。原仓库讨论两种模式：

| 模式 | 入口与可达性 | 主要代价 |
| --- | --- | --- |
| model-invoked | 保留面向模型的触发描述，可自动选择，也可由人点名；其他技能可到达 | 持续上下文负担 |
| user-invoked | 只由人显式发起，描述面向人解释用途 | 人需要记住何时调用 |

模型可调用并不排除用户手动调用；description增加的是发现途径。只有Agent需要自己选择，或其他技能确需访问这份参考，才值得承担常驻描述成本。

原文在手动模式使用 `disable-model-invocation: true`；description仍可存在，但改成简短的人类摘要，不再堆触发列表。不是删除整个description字段。

以下为按原文机制整理的阅读示例：

模型可选择的模式，描述写具体任务与触发分支：

```yaml
---
name: report-review
description: Review order-investigation reports for evidence, uncertainty, and workflow-state consistency.
---
```

仅手动的模式，明确关闭模型调用，描述面向人：

```yaml
---
name: report-review
description: Review an order-investigation report.
disable-model-invocation: true
---
```

这些是该仓库讨论的调用约定，不能当作所有Agent产品均支持的统一字段。迁移到具体客户端时核对其支持的配置和实际发现行为，不因YAML可解析就宣称自动调用设置已生效。

## 22.12 何时拆成独立技能，何时只拆参考文件

通用写作部分按顺序拆文档，技能机制部分又提出按调用需求拆：若某个独立概念在真实提示词中经常出现，需要单独触发，或者其他技能必须能调用它，才考虑新增model-invoked技能。

新增技能会增加常驻description，这个成本要由独立可达性的收益承担。如果只是某一个流程的可选资料，放普通参考文件并由条件指针访问通常已经足够。

多个自动可达技能共享参考时，可把参考集中成一个能调用的技能，减少含义复制。两个仅手动技能都需要的资料则放到普通文件，由两者分别指向；原文机制下，它们不能相互自动调用。

## 22.13 Router skill：帮助人选择入口

手动技能多到记不住时，可以有一个同样手动调用的router，列出其他技能的名字及适用情形。人只记一个入口，先取得建议，再显式选择业务技能。

这里router负责索引与提示，不代表它能自动启动仅手动的目标。原文明确保留人的调用动作，不能把路由写成能够绕过目标调用设置的自动调度器。

解释示例：router根据“设计任务、实现任务、审查修改”推荐不同入口；它说明选择理由和名称，实际发起仍由人决定。真正自动串接需要工具和客户端支持另一种执行机制，不由这份索引文字凭空获得。

## 22.14 agents/openai.yaml保存什么

本次源仓库该文件只有两项界面信息：

```yaml
interface:
  display_name: "Writing for Agents"
  short_description: "Write documents agents consume"
```

它说明显示名和短介绍，不是业务流程，也没有把通用写作规则放在这里。主文件仍负责内容和入口描述，SKILL-MECHANICS负责技能专属调用取舍。

本机插件打包版还可能增加客户端元数据，和源仓库文件不是完全相同的内容。本篇以固定源仓库版本为依据，不把打包器附加字段当原文。

## 22.15 用订单Skill检查一次文档设计

下面将原文原则对应到前面订单案例，属于帮助理解的例子，不是新增一套已安装技能。

1. 明确触发是交易核对报告，输入为约定五张CSV，而非所有办公任务。
2. 主文件保留读输入、清洗、聚合关联、Excel、Word和复核顺序。
3. 业务口径在参考资料有一个权威定义，模板和步骤只指向它。
4. 同一规则的定义、边界和错误分支相邻，例如支付互斥标签及不能确认扣款的限制。
5. 每阶段有完成证据：清洗守恒、关联粒度、实际公式值、跨文件一致性。
6. 各种输出按本批结果变化，不能固定6352或79。
7. 只保留改变执行行为的要求，重复提醒和可从环境直接读取的事实按需删减。
8. 描述、正文或资源修改后，用原样本和变化输入分别检查发现、执行与结果。

面向学习者的Notebook会保留推导、例子和解释，帮助人理解；真正写给Agent执行的主文件，应从这些知识里选择当前任务必需的步骤与指针。学习笔记更详细与Agent指令更聚焦，服务的是不同阅读需求。